# Bengali Toxicity — v12 (strengthened base+large, multi-seed)

The winning combo from v10 (0.573): **BanglaBERT-base + BanglaBERT-large**, no MuRIL. Strengthened: **max_len 160**, large **4 folds**, **2 seeds averaged per model**, OOF-weighted blend + thresholds. Official mount, test.csv IDs. ~3-4h (still one notebook, finishes). **GPU + Internet ON, Run All.**


In [ ]:
# ---- 1. Install ----
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","iterative-stratification==0.1.7"], check=False)
print("deps done")

In [ ]:
# ---- 2. Imports & CONFIG ----
import os, re, gc, glob, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, logging as hf_logging
from torch.optim import AdamW
hf_logging.set_verbosity_error()
# each model: (name, folds, batch, grad_accum, lr, epochs)
MODELS=[
    dict(name="csebuetnlp/banglabert",       folds=5, batch=16, accum=1, lr=2e-5,   epochs=4),
    dict(name="csebuetnlp/banglabert_large", folds=4, batch=8,  accum=2, lr=1.5e-5, epochs=4),
]
class CFG:
    max_len=160; lstm_hidden=256; dropout=0.3; weight_decay=0.01; warmup_ratio=0.1
    pos_weight_clip=30.0; GATE=True; seed=42; use_fp16=True
LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed); print("Device:",DEVICE)

In [ ]:
# ---- 3. Metric helpers ----
def gate_preds(prob,thr,gate=True):
    pred=(prob>=thr).astype(int)
    if gate: pred[pred[:,0]==0,1:]=0
    return pred
def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    Bv=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+Bv)/2.0,A,Bv
def tune_thresholds(y,prob):
    grid=np.arange(0.10,0.90,0.01); thr=np.full(N_LABELS,0.5)
    bt,bs=0.5,-1
    for t in grid:
        s=f1_score(y[:,0],(prob[:,0]>=t).astype(int),average="macro",zero_division=0)
        if s>bs: bs,bt=s,t
    thr[0]=bt
    for i in range(1,N_LABELS):
        bt,bs=0.5,-1
        for t in grid:
            s=f1_score(y[:,i],(prob[:,i]>=t).astype(int),average="binary",zero_division=0)
            if s>bs: bs,bt=s,t
        thr[i]=bt if bs>0 else 0.5
    return thr

In [ ]:
# ---- 4. Load data (official mount; test.csv IDs) ----
def find_data_dir():
    for c in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026","/kaggle/input/bengali-nlp-safety-challenge-2026"]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    hits=glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits: return os.path.dirname(sorted(hits,key=len)[0])
    for c in ["./data","."]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    raise FileNotFoundError("Attach the competition dataset.")
DATA_DIR=find_data_dir(); print("Data dir:",DATA_DIR)
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv")); test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
try: val_df=pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
except FileNotFoundError: val_df=None
def pick(df,pref,kw):
    for p in pref:
        if p in df.columns: return p
    for c in df.columns:
        if any(k in c.lower() for k in kw): return c
    o=[c for c in df.columns if df[c].dtype==object]; return max(o,key=lambda c: df[c].astype(str).str.len().mean())
TEXT_COL=pick(train_df,["text"],["text","comment"]); ID_COL=pick(test_df,["ID","id"],["id"])
full_df=pd.concat([train_df,val_df],ignore_index=True) if (val_df is not None and set(LABELS).issubset(val_df.columns)) else train_df.copy()
full_df=full_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
for l in LABELS: full_df[l]=full_df[l].astype(int)
print("Full",full_df.shape,"Test",test_df.shape)

In [ ]:
# ---- 5. Cleaning (proven simple) ----
def clean_text(t):
    if not isinstance(t,str): return ""
    t=re.sub(r"http\S+|www\.\S+","",t); t=re.sub(r"<.*?>","",t); t=re.sub(r"@\w+","",t)
    t=re.sub(r"\s+"," ",t).strip(); return t.replace("#","")
full_df["clean"]=full_df[TEXT_COL].apply(clean_text); test_df["clean"]=test_df[TEXT_COL].apply(clean_text)
print(full_df[["clean"]].head(2).to_string())

In [ ]:
# ---- 6. Dataset, Model, pos_weight ----
class DS(Dataset):
    def __init__(s,texts,tok,labels=None): s.t=list(texts); s.tok=tok; s.y=labels
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=s.tok(str(s.t[i]),add_special_tokens=True,max_length=CFG.max_len,padding="max_length",
                truncation=True,return_token_type_ids=False,return_attention_mask=True,return_tensors="pt")
        it={"input_ids":e["input_ids"].flatten(),"attention_mask":e["attention_mask"].flatten()}
        if s.y is not None: it["labels"]=torch.tensor(s.y[i],dtype=torch.float)
        return it
class BertBiLSTM(nn.Module):
    def __init__(s,name,n_out=7):
        super().__init__()
        s.bert=AutoModel.from_pretrained(name)
        s.lstm=nn.LSTM(s.bert.config.hidden_size,CFG.lstm_hidden,num_layers=2,batch_first=True,bidirectional=True,dropout=CFG.dropout)
        s.dropout=nn.Dropout(CFG.dropout); s.fc=nn.Linear(CFG.lstm_hidden*2,n_out)
    def forward(s,input_ids,attention_mask):
        seq=s.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        lo,_=s.lstm(seq); m=attention_mask.unsqueeze(-1).float()
        pooled=(lo*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return s.fc(s.dropout(pooled))
def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(neg/np.clip(pos,1,None),1.0,CFG.pos_weight_clip)  # is_toxic auto ->1.0 (not upweighted)
print("model ready")

In [ ]:
# ---- 7. Train one fold (per-model cfg) ----
@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_fp16): lg=model(ids,mask)
        out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out,0)
def train_fold(mc,tok,trt,trY,vat,vaY,tet,fold):
    seed_everything(CFG.seed+fold)
    model=BertBiLSTM(mc["name"],N_LABELS).to(DEVICE)
    trl=DataLoader(DS(trt,tok,trY),batch_size=mc["batch"],shuffle=True,num_workers=0,pin_memory=True)
    vl=DataLoader(DS(vat,tok,vaY),batch_size=64,shuffle=False,num_workers=0)
    el=DataLoader(DS(tet,tok),batch_size=64,shuffle=False,num_workers=0)
    pw=torch.tensor(pos_weight_from(trY),dtype=torch.float,device=DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    steps=max(1,(len(trl)//mc["accum"]))*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(steps*CFG.warmup_ratio),steps)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG.use_fp16); best,bv,bt=-1,None,None
    for ep in range(mc["epochs"]):
        model.train(); run=0; opt.zero_grad()
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE); y=b["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG.use_fp16): loss=crit(model(ids,mask),y)/mc["accum"]
            scaler.scale(loss).backward(); run+=loss.item()*mc["accum"]
            if (st+1)%mc["accum"]==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        vp=predict(model,vl); th=tune_thresholds(vaY,vp); c,A,Bv=combined_score(vaY,gate_preds(vp,th,CFG.GATE))
        print(f"    {mc['name'].split('/')[-1]} fold{fold} ep{ep+1}/{mc['epochs']} loss={run/len(trl):.4f} valC={c:.4f} (A={A:.3f} B={Bv:.3f})")
        if c>=best: best=c; bv=vp; bt=predict(model,el)
    del model; gc.collect(); torch.cuda.empty_cache()
    return bv,bt

In [ ]:
# ---- 8. Train each model with its own K-fold; collect OOF + test ----
from sklearn.model_selection import StratifiedKFold
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    HAVE_ITER=True
except Exception: HAVE_ITER=False
X=full_df["clean"].values; Y=full_df[LABELS].values.astype(float); test_texts=test_df["clean"].values
def make_folds(nf):
    if HAVE_ITER:
        return list(MultilabelStratifiedKFold(n_splits=nf,shuffle=True,random_state=42).split(X,Y.astype(int)))
    return list(StratifiedKFold(n_splits=nf,shuffle=True,random_state=42).split(X,Y[:,0].astype(int)))
SEEDS=[42,1337]
oofs=[]; tests=[]
for mc in MODELS:
    print(f"########## {mc['name']} ({mc['folds']} folds x {len(SEEDS)} seeds) ##########")
    tok=AutoTokenizer.from_pretrained(mc["name"])
    oof=np.zeros((len(full_df),N_LABELS)); tp=np.zeros((len(test_df),N_LABELS))
    for si,sd in enumerate(SEEDS):
        for fold,(tr,va) in enumerate(make_folds(mc["folds"])):
            vp,tt=train_fold(mc,tok,X[tr],Y[tr],X[va],Y[va],test_texts,fold*10+si)
            oof[va]+=vp/len(SEEDS); tp+=tt/(mc["folds"]*len(SEEDS))
    oofs.append(oof); tests.append(tp)
    c,A,Bv=combined_score(Y,gate_preds(oof,tune_thresholds(Y,oof),CFG.GATE))
    print(f"  [{mc['name'].split('/')[-1]}] OOF Combined={c:.4f} (A={A:.3f} B={Bv:.3f})")

In [ ]:
# ---- 9. Blend (weight tuned on OOF) + thresholds ----
best=(-1,None,None)
for w in np.round(np.arange(0,1.01,0.05),2):
    ob=w*oofs[0]+(1-w)*oofs[1]; th=tune_thresholds(Y,ob)
    c,_,_=combined_score(Y,gate_preds(ob,th,CFG.GATE))
    if c>best[0]: best=(c,w,th)
_,bw,thr=best
oof=bw*oofs[0]+(1-bw)*oofs[1]; test_pred=bw*tests[0]+(1-bw)*tests[1]
np.save("oof.npy",oof); np.save("test_pred.npy",test_pred)
op=gate_preds(oof,thr,CFG.GATE); c,A,Bv=combined_score(Y,op)
print("="*54)
print(f"Blend weight base={bw} large={round(1-bw,2)}")
print(f"OOF Subtask A={A:.4f} | Subtask B={Bv:.4f}")
print(f"OOF COMBINED SCORE = {c:.4f}   <-- expected leaderboard")
print("="*54)
for i,l in enumerate(LABELS):
    ff=f1_score(Y[:,i],op[:,i],average=("macro" if i==0 else "binary"),zero_division=0)
    print(f"  {l:<20} thr={thr[i]:.2f} F1={ff:.4f}")

In [ ]:
# ---- 10. Submission (test.csv IDs) ----
final=gate_preds(test_pred,thr,CFG.GATE)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].values})
for i,l in enumerate(LABELS): sub[l]=final[:,i]
sub.to_csv("submission.csv",index=False)
print("submission.csv",sub.shape,"| IDs from official test.csv"); print(sub.head())